# Plugin Spec
> Pluggy hookspecs and session ABCs

In [ ]:
#| default_exp pluginspec

In [ ]:
#| export
import pluggy
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Optional, List

hookspec = pluggy.HookspecMarker('memexcode')
hookimpl = pluggy.HookimplMarker('memexcode')

In [ ]:
#| export

@dataclass
class SessionInfo:
    id: str
    cwd: str
    created_at: str
    updated_at: str
    name: Optional[str] = None
    parent_session: Optional[str] = None

In [ ]:
#| export

class SessionStoreHookSpec:
    @hookspec
    def get_store(self):
        '''Return (name, StoreClass) tuple for this store implementation.'''

In [ ]:
#| export

class SessionABC(ABC):
    @abstractmethod
    async def append(self, entry: "SessionEntry") -> None:
        '''Persist one entry.'''
        ...

    @abstractmethod
    async def get_by_id(self, entry_id: str) -> Optional["SessionEntry"]:
        '''Random access by id.'''
        ...

    @abstractmethod
    async def get_path_to_root(self, leaf_id: str) -> List["SessionEntry"]:
        '''Walk parentId chain from leaf back to root. Ordered [root..leaf].'''
        ...

    @abstractmethod
    async def load_all(self) -> List["SessionEntry"]:
        '''All entries in insertion order.'''
        ...

    @abstractmethod
    async def compact(self, first_kept_id: str, summary: str) -> None:
        '''Rewrite history: replace entries before first_kept_id with a compaction entry.'''
        ...

    @abstractmethod
    async def get_tree(self, root_id: Optional[str] = None) -> 'TreeNode':
        '''Get full tree from root.'''
        ...

In [ ]:
#| export

@dataclass
class TreeNode:
    entry: "SessionEntry"
    children: List['TreeNode']

In [ ]:
#| export

class SessionCollectionABC(ABC):
    @abstractmethod
    async def create(self, cwd: str, name: Optional[str] = None, parent_session: Optional[str] = None) -> SessionInfo:
        '''Create a new session.'''
        ...

    @abstractmethod
    async def list(self) -> List[SessionInfo]:
        '''List all sessions.'''
        ...

    @abstractmethod
    async def rename(self, session_id: str, name: str) -> None:
        '''Rename a session.'''
        ...

    @abstractmethod
    async def delete(self, session_id: str) -> None:
        '''Delete a session.'''
        ...

    @abstractmethod
    async def fork(self, session_id: str, leaf_id: Optional[str] = None, name: Optional[str] = None) -> SessionInfo:
        '''Fork a session at a given leaf.'''
        ...

In [ ]:
#| export

class SessionStore(SessionCollectionABC):
    @abstractmethod
    def get_session(self, session_id: str) -> SessionABC:
        '''Get a per-session store for a given session id.'''
        ...

In [ ]:
#| export
from typing import TYPE_CHECKING
if TYPE_CHECKING:
    from memexcode.session import SessionEntry